In [1]:
# ==========================================================
# Imports
# ==========================================================

import os
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple

import cv2
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

c:\Users\utkar\Desktop\Study\Projects\Object_Detection_Using_Tensorflow\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================================
# Suppress Warnings
# ==========================================================

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [3]:
# ==========================================================
# Library Versions
# ==========================================================

print("=" * 60)
print("Environment Information")
print("=" * 60)

print(f"Python      : {os.sys.version.split()[0]}")
print(f"TensorFlow  : {tf.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print(f"Numpy       : {np.__version__}")

Environment Information
Python      : 3.11.9
TensorFlow  : 2.18.0
OpenCV      : 5.0.0
Numpy       : 2.0.2


In [4]:
# ==========================================================
# GPU Configuration
# ==========================================================

gpus = tf.config.list_physical_devices("GPU")
GPU_DEVICE = "/GPU:0" if gpus else "/CPU:0"

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        print(f"GPU Available : {len(gpus)}")
        print(f"Using Device  : {GPU_DEVICE}")
        print(gpus[0].name)

    except RuntimeError as e:
        print(e)

else:
    print("Running on CPU")
    print(f"Using Device  : {GPU_DEVICE}")

Running on CPU
Using Device  : /CPU:0


In [5]:
# ==========================================================
# Random Seed
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [6]:
# ==========================================================
# Detection Configuration
# ==========================================================

GRID_SIZE = 14
NUM_BOX_ATTRIBUTES = 4

In [7]:
# ==========================================================
# Project Configuration
# ==========================================================

PROJECT_ROOT = Path.cwd()

DATASET_ROOT = PROJECT_ROOT / "dataset"

TRAIN_DIR = DATASET_ROOT / "train"
VALID_DIR = DATASET_ROOT / "valid"
TEST_DIR = DATASET_ROOT / "test"

TRAIN_IMAGE_DIR = TRAIN_DIR / "images"
VALID_IMAGE_DIR = VALID_DIR / "images"
TEST_IMAGE_DIR = TEST_DIR / "images"

TRAIN_JSON = TRAIN_DIR / "_annotations.coco.json"
VALID_JSON = VALID_DIR / "_annotations.coco.json"
TEST_JSON = TEST_DIR / "_annotations.coco.json"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

IMAGE_SIZE = (224, 224)

NUM_CHANNELS = 3
INPUT_SHAPE = (IMAGE_SIZE[0], IMAGE_SIZE[1], NUM_CHANNELS)
NUM_CLASSES = 10

BATCH_SIZE = 16
EPOCHS = 25

LEARNING_RATE = 1e-3

CONFIDENCE_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5

In [8]:
# ==========================================================
# Create Project Directories
# ==========================================================

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

In [9]:
# ==========================================================
# Validate Dataset Structure
# ==========================================================

required_paths = [
    TRAIN_IMAGE_DIR,
    VALID_IMAGE_DIR,
    TEST_IMAGE_DIR,
    TRAIN_JSON,
    VALID_JSON,
    TEST_JSON
]

missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    print("Missing Files/Folders:\n")
    for path in missing_paths:
        print(path)
else:
    print("Dataset structure verified.")

Dataset structure verified.


In [10]:
# ==========================================================
# Validate Dataset Structure
# ==========================================================

required_paths = [
    TRAIN_IMAGE_DIR,
    VALID_IMAGE_DIR,
    TEST_IMAGE_DIR,
    TRAIN_JSON,
    VALID_JSON,
    TEST_JSON
]

missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    print("Missing Files/Folders:\n")
    for path in missing_paths:
        print(path)
else:
    print("Dataset structure verified.")

Dataset structure verified.


In [11]:
# ==========================================================
# Utility Functions
# ==========================================================

def load_json(json_path: Path) -> Dict:
    """Load a JSON annotation file."""
    with open(json_path, "r") as file:
        return json.load(file)


def load_image(image_path: Path) -> np.ndarray:
    """Load an RGB image."""
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image


def resize_image(image: np.ndarray, size: Tuple[int, int]) -> np.ndarray:
    """Resize an image."""
    return cv2.resize(image, size)


def normalize_image(image: np.ndarray) -> np.ndarray:
    """Normalize image to [0,1]."""
    return image.astype(np.float32) / 255.0


def save_model(model: tf.keras.Model, path: Path) -> None:
    """Save TensorFlow model."""
    model.save(path)


def load_model(path: Path) -> tf.keras.Model:
    """Load TensorFlow model."""
    return tf.keras.models.load_model(path)

In [12]:
# ==========================================================
# Configuration Summary
# ==========================================================

print("=" * 60)
print("Configuration")
print("=" * 60)

print(f"Dataset Root      : {DATASET_ROOT}")
print(f"Image Size        : {IMAGE_SIZE}")
print(f"Batch Size        : {BATCH_SIZE}")
print(f"Epochs            : {EPOCHS}")
print(f"Learning Rate     : {LEARNING_RATE}")
print(f"Number of Classes : {NUM_CLASSES}")

Configuration
Dataset Root      : c:\Users\utkar\Desktop\Study\Projects\Object_Detection_Using_Tensorflow\notebooks\dataset
Image Size        : (224, 224)
Batch Size        : 16
Epochs            : 25
Learning Rate     : 0.001
Number of Classes : 10


In [13]:
# ==========================================================
# COCO Dataset Parser
# ==========================================================

class COCODataset:

    def __init__(self, image_dir: Path, annotation_file: Path):

        self.image_dir = image_dir
        self.annotation_file = annotation_file

        self.data = load_json(annotation_file)

        self.images = self.data["images"]
        self.annotations = self.data["annotations"]
        self.categories = self.data["categories"]

        self.image_map = {}
        self.annotation_map = defaultdict(list)
        self.category_map = {}
        self.category_name_to_id = {}

        self._build_indices()

    def _build_indices(self) -> None:

        for image in self.images:
            self.image_map[image["id"]] = image

        for annotation in self.annotations:
            self.annotation_map[annotation["image_id"]].append(annotation)

        for category in self.categories:
            self.category_map[category["id"]] = category["name"]
            self.category_name_to_id[category["name"]] = category["id"]

    def __len__(self) -> int:
        return len(self.images)

    def get_image_info(self, index: int) -> Dict:
        return self.images[index]

    def load_image(self, index: int) -> np.ndarray:
        image_info = self.images[index]
        image_path = self.image_dir / image_info["file_name"]
        return load_image(image_path)

    def load_annotations(self, index: int) -> List[Dict]:
        image_id = self.images[index]["id"]
        return self.annotation_map.get(image_id, [])

    def __getitem__(self, index: int):

        image = self.load_image(index)
        image_info = self.get_image_info(index)
        annotations = self.load_annotations(index)

        return image, image_info, annotations

    def summary(self) -> None:

        print("=" * 50)
        print("Dataset Summary")
        print("=" * 50)

        print(f"Images      : {len(self.images)}")
        print(f"Annotations : {len(self.annotations)}")
        print(f"Classes     : {len(self.categories)}")

In [14]:
# ==========================================================
# Load Dataset
# ==========================================================

train_dataset = COCODataset(TRAIN_IMAGE_DIR, TRAIN_JSON)
valid_dataset = COCODataset(VALID_IMAGE_DIR, VALID_JSON)
test_dataset = COCODataset(TEST_IMAGE_DIR, TEST_JSON)

In [15]:
print("Training Dataset")
train_dataset.summary()

print()

print("Validation Dataset")
valid_dataset.summary()

print()

print("Test Dataset")
test_dataset.summary()

Training Dataset
Dataset Summary
Images      : 6228
Annotations : 37362
Classes     : 11

Validation Dataset
Dataset Summary
Images      : 623
Annotations : 3735
Classes     : 11

Test Dataset
Dataset Summary
Images      : 310
Annotations : 1860
Classes     : 11


In [16]:
print("=" * 50)
print("Classes")
print("=" * 50)

for class_id, class_name in train_dataset.category_map.items():
    print(f"{class_id:>2} -> {class_name}")

Classes
 0 -> digits
 1 -> 0
 2 -> 1
 3 -> 2
 4 -> 3
 5 -> 4
 6 -> 5
 7 -> 6
 8 -> 7
 9 -> 8
10 -> 9


In [17]:
image, image_info, annotations = train_dataset[0]

print("Image Information")
print(image_info)

print()

print("Annotations")

for ann in annotations:
    print(ann)

Image Information
{'id': 0, 'license': 1, 'file_name': '361361_jpg.rf.00094c3f3470a0fbe10b14120a3fc1af.jpg', 'height': 100, 'width': 200, 'date_captured': '2024-05-01T07:13:16+00:00'}

Annotations
{'id': 0, 'image_id': 0, 'category_id': 4, 'bbox': [6, 22, 29.128, 52.722], 'area': 1535.675, 'segmentation': [], 'iscrowd': 0}
{'id': 1, 'image_id': 0, 'category_id': 7, 'bbox': [39, 25, 26.063, 43.11], 'area': 1123.554, 'segmentation': [], 'iscrowd': 0}
{'id': 2, 'image_id': 0, 'category_id': 2, 'bbox': [64, 19, 26.39, 49.095], 'area': 1295.617, 'segmentation': [], 'iscrowd': 0}
{'id': 3, 'image_id': 0, 'category_id': 4, 'bbox': [92, 22, 28.798, 52.798], 'area': 1520.436, 'segmentation': [], 'iscrowd': 0}
{'id': 4, 'image_id': 0, 'category_id': 7, 'bbox': [124, 30, 28.97, 49.032], 'area': 1420.472, 'segmentation': [], 'iscrowd': 0}
{'id': 5, 'image_id': 0, 'category_id': 2, 'bbox': [148, 26, 28.125, 42.048], 'area': 1182.586, 'segmentation': [], 'iscrowd': 0}


In [18]:
# ==========================================================
# Dataset Statistics
# ==========================================================

def find_max_objects(dataset: COCODataset) -> int:
    """Return the maximum number of objects present in a single image."""
    return max(len(dataset.annotation_map[image["id"]]) for image in dataset.images)


MAX_OBJECTS = find_max_objects(train_dataset)

print(f"Maximum Objects per Image : {MAX_OBJECTS}")

Maximum Objects per Image : 6


In [19]:
# ==========================================================
# Bounding Box Utilities
# ==========================================================

def resize_boxes(
    boxes: np.ndarray,
    original_size: Tuple[int, int],
    target_size: Tuple[int, int]
) -> np.ndarray:
    """Resize bounding boxes to match resized image."""

    orig_h, orig_w = original_size
    target_h, target_w = target_size

    scale_x = target_w / orig_w
    scale_y = target_h / orig_h

    boxes = boxes.copy().astype(np.float32)

    boxes[:, 0] *= scale_x
    boxes[:, 1] *= scale_y
    boxes[:, 2] *= scale_x
    boxes[:, 3] *= scale_y

    return boxes

In [20]:
# ==========================================================
# Grid Encoder
# ==========================================================

def encode_grid(
    annotations: List[Dict],
    original_size: Tuple[int, int],
    target_size: Tuple[int, int]
):

    target_h, target_w = target_size
    orig_h, orig_w = original_size

    scale_x = target_w / orig_w
    scale_y = target_h / orig_h

    grid_boxes = np.zeros((GRID_SIZE, GRID_SIZE, 4), dtype=np.float32)

    grid_objectness = np.zeros(
        (GRID_SIZE, GRID_SIZE, 1),
        dtype=np.float32
    )

    grid_classes = np.zeros(
        (GRID_SIZE, GRID_SIZE, NUM_CLASSES),
        dtype=np.float32
    )

    cell_w = target_w / GRID_SIZE
    cell_h = target_h / GRID_SIZE

    for ann in annotations:

        x, y, w, h = ann["bbox"]

        if w <= 0 or h <= 0:
            continue

        x1 = x * scale_x
        y1 = y * scale_y

        x2 = (x + w) * scale_x
        y2 = (y + h) * scale_y

        x_center = ((x1 + x2) / 2) / target_w
        y_center = ((y1 + y2) / 2) / target_h

        box_width = (x2 - x1) / target_w
        box_height = (y2 - y1) / target_h

        col = int((x_center * target_w) / cell_w)
        row = int((y_center * target_h) / cell_h)

        row = np.clip(row, 0, GRID_SIZE - 1)
        col = np.clip(col, 0, GRID_SIZE - 1)

        grid_boxes[row, col] = [
            x_center,
            y_center,
            box_width,
            box_height
        ]

        grid_objectness[row, col, 0] = 1.0

        class_id = ann["category_id"]

        if 0 <= class_id < NUM_CLASSES:
            grid_classes[row, col, class_id] = 1.0

    return grid_boxes, grid_objectness, grid_classes

In [21]:
# ==========================================================
# Sample Preprocessing
# ==========================================================

def preprocess_sample(
    image: np.ndarray,
    annotations: List[Dict]
):

    original_size = image.shape[:2]

    image = resize_image(image, IMAGE_SIZE)

    image = normalize_image(image)

    boxes, objectness, classes = encode_grid(
        annotations,
        original_size,
        IMAGE_SIZE
    )

    return image, boxes, objectness, classes

In [22]:
# ==========================================================
# Dataset Preprocessing
# ==========================================================

def preprocess_dataset(dataset: COCODataset):

    images = []
    boxes = []
    objectness = []
    classes = []

    for i in tqdm(range(len(dataset))):

        image, _, annotations = dataset[i]

        image, box, obj, cls = preprocess_sample(
            image,
            annotations
        )

        images.append(image)
        boxes.append(box)
        objectness.append(obj)
        classes.append(cls)

    return (
        np.asarray(images, dtype=np.float32),
        np.asarray(boxes, dtype=np.float32),
        np.asarray(objectness, dtype=np.float32),
        np.asarray(classes, dtype=np.float32)
    )

In [23]:
# ==========================================================
# Bounding Box Decoder
# ==========================================================

def decode_box(box, image_size=IMAGE_SIZE):

    h, w = image_size

    x_center, y_center, bw, bh = box

    x_center *= w
    y_center *= h

    bw *= w
    bh *= h

    x1 = x_center - bw / 2
    y1 = y_center - bh / 2

    x2 = x_center + bw / 2
    y2 = y_center + bh / 2

    return np.array([x1, y1, x2, y2], dtype=np.float32)

In [24]:
train_images, train_boxes, train_objectness, train_classes = preprocess_dataset(train_dataset)
valid_images, valid_boxes, valid_objectness, valid_classes = preprocess_dataset(valid_dataset)
test_images, test_boxes, test_objectness, test_classes = preprocess_dataset(test_dataset)

100%|██████████| 310/310 [00:00<00:00, 1312.93it/s]


In [25]:
def create_tf_dataset(
    images,
    boxes,
    objectness,
    classes,
    batch_size,
    shuffle=False
):

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            images,
            {
                "boxes": boxes,
                "objectness": objectness,
                "classes": classes
            }
        )
    )

    if shuffle:
        dataset = dataset.shuffle(
            len(images),
            seed=SEED
        )

    dataset = dataset.batch(batch_size)

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset

In [26]:
train_tf_dataset = create_tf_dataset(
    train_images,
    train_boxes,
    train_objectness,
    train_classes,
    BATCH_SIZE,
    shuffle=True
)

valid_tf_dataset = create_tf_dataset(
    valid_images,
    valid_boxes,
    valid_objectness,
    valid_classes,
    BATCH_SIZE
)

test_tf_dataset = create_tf_dataset(
    test_images,
    test_boxes,
    test_objectness,
    test_classes,
    BATCH_SIZE
)

In [27]:
print(train_images.shape)
print(train_boxes.shape)
print(train_objectness.shape)
print(train_classes.shape)

(6228, 224, 224, 3)
(6228, 14, 14, 4)
(6228, 14, 14, 1)
(6228, 14, 14, 10)


In [28]:
# ==========================================================
# CNN Building Blocks
# ==========================================================

def conv_block(
    inputs: tf.Tensor,
    filters: int,
    kernel_size: int = 3,
    pool: bool = True
) -> tf.Tensor:
    """Create a convolution block."""

    x = tf.keras.layers.Conv2D(
        filters=filters,
        kernel_size=kernel_size,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal"
    )(inputs)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    if pool:
        x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    return x

In [29]:
# ==========================================================
# CNN Backbone
# ==========================================================

def build_backbone(
    input_shape: Tuple[int, int, int]
) -> tf.keras.Model:
    """Build CNN feature extractor."""

    inputs = tf.keras.Input(shape=input_shape, name="image")

    x = conv_block(inputs, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = conv_block(x, 256)
    outputs = conv_block(x, 512, pool=False)

    return tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="CNN_Backbone"
    )

In [30]:
with tf.device(GPU_DEVICE):
    backbone = build_backbone(INPUT_SHAPE)

In [31]:
backbone.summary()

total_params = backbone.count_params()
trainable_params = sum(np.prod(v.shape) for v in backbone.trainable_weights)
non_trainable_params = sum(np.prod(v.shape) for v in backbone.non_trainable_weights)

print("=" * 60)
print("Backbone Statistics")
print("=" * 60)
print(f"Total Parameters        : {total_params:,}")
print(f"Trainable Parameters    : {trainable_params:,}")
print(f"Non-Trainable Parameters: {non_trainable_params:,}")

Model: "CNN_Backbone"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       294,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 14, 14, 512)    │     1,179,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 14, 14, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 14, 14, 512)    │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,571,552 (5.99 MB)

 Trainable params: 1,569,568 (5.99 MB)

 Non-trainable params: 1,984 (7.75 KB)

Backbone Statistics
Total Parameters        : 1,571,552
Trainable Parameters    : 1,569,568
Non-Trainable Parameters: 1,984


In [32]:
sample = tf.random.normal((1, *INPUT_SHAPE))
features = backbone(sample)
print("Feature Shape :", features.shape)

Feature Shape : (1, 14, 14, 512)


In [33]:
print("=" * 60)
print("Backbone Output")
print("=" * 60)

print(f"Input Shape  : {INPUT_SHAPE}")
print(f"Output Shape : {features.shape}")

Backbone Output
Input Shape  : (224, 224, 3)
Output Shape : (1, 14, 14, 512)


In [34]:
# ==========================================================
# Detection Head
# ==========================================================

def build_detector(backbone: tf.keras.Model) -> tf.keras.Model:
    """Build fully convolutional object detector."""

    inputs = backbone.input

    x = backbone.output

    x = tf.keras.layers.Conv2D(
        filters=256,
        kernel_size=3,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal"
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    box_output = tf.keras.layers.Conv2D(
        filters=4,
        kernel_size=1,
        padding="same",
        name="boxes"
    )(x)

    object_output = tf.keras.layers.Conv2D(
        filters=1,
        kernel_size=1,
        activation="sigmoid",
        padding="same",
        name="objectness"
    )(x)

    class_output = tf.keras.layers.Conv2D(
        filters=NUM_CLASSES,
        kernel_size=1,
        activation="softmax",
        padding="same",
        name="classes"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs={
            "boxes": box_output,
            "objectness": object_output,
            "classes": class_output
        },
        name="DigitDetector"
    )

    return model

In [35]:
with tf.device(GPU_DEVICE):
    detector = build_detector(backbone)

In [36]:
detector.summary()

Model: "DigitDetector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        864 │ image[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 224, 224,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ re_lu[0][0]       │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │     18,432 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ re_lu_1[0][0]     │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │     73,728 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 56, 56,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ re_lu_2[0][0]     │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 28, 28,    │    294,912 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │      1,024 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 28, 28,    │          0 │ batch_normalizat… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 14, 14,    │          0 │ re_lu_3[0][0]   

 Total params: 2,756,079 (10.51 MB)

 Trainable params: 2,753,583 (10.50 MB)

 Non-trainable params: 2,496 (9.75 KB)

In [37]:
sample = tf.random.normal((1, *INPUT_SHAPE))

prediction = detector(sample)

print(prediction["boxes"].shape)
print(prediction["objectness"].shape)
print(prediction["classes"].shape)

(1, 14, 14, 4)
(1, 14, 14, 1)
(1, 14, 14, 10)


In [38]:
total_params = detector.count_params()
trainable_params = sum(np.prod(v.shape) for v in detector.trainable_weights)
non_trainable_params = sum(np.prod(v.shape) for v in detector.non_trainable_weights)

print("=" * 60)
print("Detector Statistics")
print("=" * 60)
print(f"Total Parameters        : {total_params:,}")
print(f"Trainable Parameters    : {trainable_params:,}")
print(f"Non-Trainable Parameters: {non_trainable_params:,}")

Detector Statistics
Total Parameters        : 2,756,079
Trainable Parameters    : 2,753,583
Non-Trainable Parameters: 2,496


In [39]:
sample_boxes = train_boxes[0]
sample_objectness = train_objectness[0]
sample_classes = train_classes[0]

indices = np.argwhere(sample_objectness[..., 0] == 1)

print(f"Objects Found: {len(indices)}")

for row, col in indices:

    print("-" * 50)
    print(f"Grid Cell : ({row}, {col})")
    print(f"Box       : {sample_boxes[row, col]}")
    print(f"Class     : {np.argmax(sample_classes[row, col])}")

Objects Found: 6
--------------------------------------------------
Grid Cell : (6, 1)
Box       : [0.10282 0.48361 0.14564 0.52722]
Class     : 4
--------------------------------------------------
Grid Cell : (6, 3)
Box       : [0.2601575 0.46555   0.130315  0.4311   ]
Class     : 7
--------------------------------------------------
Grid Cell : (6, 5)
Box       : [0.385975 0.435475 0.13195  0.49095 ]
Class     : 2
--------------------------------------------------
Grid Cell : (6, 7)
Box       : [0.531995 0.48399  0.14399  0.52798 ]
Class     : 4
--------------------------------------------------
Grid Cell : (6, 11)
Box       : [0.8103125 0.47024   0.140625  0.42048  ]
Class     : 2
--------------------------------------------------
Grid Cell : (7, 9)
Box       : [0.692425 0.54516  0.14485  0.49032 ]
Class     : 7


In [40]:
# ==========================================================
# Bounding Box Loss
# ==========================================================

bbox_loss = tf.keras.losses.Huber(
    reduction=tf.keras.losses.Reduction.NONE
)

In [42]:
# ==========================================================
# Classification Loss
# ==========================================================

classification_loss = tf.keras.losses.CategoricalCrossentropy(
    reduction=tf.keras.losses.Reduction.NONE
)

In [43]:
def compute_box_loss(y_true, y_pred, object_mask):

    loss = bbox_loss(
        y_true,
        y_pred
    )

    loss = loss * object_mask

    return tf.reduce_mean(loss)

In [44]:
def compute_objectness_loss(y_true, y_pred):

    loss = objectness_loss(
        y_true,
        y_pred
    )

    return tf.reduce_mean(loss)

In [45]:
def compute_classification_loss(y_true, y_pred, object_mask):

    loss = classification_loss(
        y_true,
        y_pred
    )

    loss = loss * object_mask

    return tf.reduce_mean(loss)

In [46]:
def detector_loss(
    y_true,
    y_pred
):

    object_mask = tf.squeeze(
        y_true["objectness"],
        axis=-1
    )

    box = compute_box_loss(
        y_true["boxes"],
        y_pred["boxes"],
        object_mask
    )

    obj = compute_objectness_loss(
        y_true["objectness"],
        y_pred["objectness"]
    )

    cls = compute_classification_loss(
        y_true["classes"],
        y_pred["classes"],
        object_mask
    )

    total = box + obj + cls

    return total

In [47]:
# ==========================================================
# Optimizer
# ==========================================================

optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

In [48]:
# ==========================================================
# Training Metrics
# ==========================================================

train_loss = tf.keras.metrics.Mean(name="train_loss")
train_box_loss = tf.keras.metrics.Mean(name="box_loss")
train_object_loss = tf.keras.metrics.Mean(name="objectness_loss")
train_class_loss = tf.keras.metrics.Mean(name="class_loss")

valid_loss = tf.keras.metrics.Mean(name="valid_loss")

In [49]:
# ==========================================================
# Training Step
# ==========================================================

@tf.function
def train_step(images, targets):
    with tf.device(GPU_DEVICE):
        with tf.GradientTape() as tape:

            predictions = detector(
                images,
                training=True
            )

            object_mask = tf.squeeze(
                targets["objectness"],
                axis=-1
            )

            box_loss = compute_box_loss(
                targets["boxes"],
                predictions["boxes"],
                object_mask
            )

            object_loss = compute_objectness_loss(
                targets["objectness"],
                predictions["objectness"]
            )

            class_loss = compute_classification_loss(
                targets["classes"],
                predictions["classes"],
                object_mask
            )

            total_loss = (
                box_loss +
                object_loss +
                class_loss
            )

    gradients = tape.gradient(
        total_loss,
        detector.trainable_variables
    )

    optimizer.apply_gradients(
        zip(
            gradients,
            detector.trainable_variables
        )
    )

    train_loss.update_state(total_loss)
    train_box_loss.update_state(box_loss)
    train_object_loss.update_state(object_loss)
    train_class_loss.update_state(class_loss)

In [50]:
# ==========================================================
# Validation Step
# ==========================================================

@tf.function
def valid_step(images, targets):
    with tf.device(GPU_DEVICE):
        predictions = detector(
            images,
            training=False
        )

        object_mask = tf.squeeze(
            targets["objectness"],
            axis=-1
        )

        box_loss = compute_box_loss(
            targets["boxes"],
            predictions["boxes"],
            object_mask
        )

        object_loss = compute_objectness_loss(
            targets["objectness"],
            predictions["objectness"]
        )

        class_loss = compute_classification_loss(
            targets["classes"],
            predictions["classes"],
            object_mask
        )

        total_loss = (
            box_loss +
            object_loss +
            class_loss
        )

    valid_loss.update_state(total_loss)

In [51]:
# ==========================================================
# Reset Metrics
# ==========================================================

def reset_metrics():

    train_loss.reset_state()
    train_box_loss.reset_state()
    train_object_loss.reset_state()
    train_class_loss.reset_state()

    valid_loss.reset_state()

In [52]:
images, targets = next(iter(train_tf_dataset))

predictions = detector(images)

print("Predicted Boxes :", predictions["boxes"].shape)
print("True Boxes      :", targets["boxes"].shape)

print("Predicted Obj   :", predictions["objectness"].shape)
print("True Obj        :", targets["objectness"].shape)

print("Predicted Class :", predictions["classes"].shape)
print("True Class      :", targets["classes"].shape)

Predicted Boxes : (16, 14, 14, 4)
True Boxes      : (16, 14, 14, 4)
Predicted Obj   : (16, 14, 14, 1)
True Obj        : (16, 14, 14, 1)
Predicted Class : (16, 14, 14, 10)
True Class      : (16, 14, 14, 10)


In [53]:
# ==========================================================
# Training Loop
# ==========================================================

for epoch in range(EPOCHS):

    reset_metrics()

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    for images, targets in train_tf_dataset:

        train_step(
            images,
            targets
        )

    for images, targets in valid_tf_dataset:

        valid_step(
            images,
            targets
        )

    print(
        f"Train Loss : {train_loss.result():.4f} | "
        f"Box : {train_box_loss.result():.4f} | "
        f"Obj : {train_object_loss.result():.4f} | "
        f"Class : {train_class_loss.result():.4f} | "
        f"Valid : {valid_loss.result():.4f}"
    )


Epoch 1/25
Train Loss : 0.1131 | Box : 0.0013 | Obj : 0.1022 | Class : 0.0096 | Valid : 0.1136

Epoch 2/25
Train Loss : 0.0461 | Box : 0.0009 | Obj : 0.0408 | Class : 0.0043 | Valid : 0.0577

Epoch 3/25
Train Loss : 0.0449 | Box : 0.0015 | Obj : 0.0382 | Class : 0.0052 | Valid : 0.0507

Epoch 4/25
Train Loss : 0.0560 | Box : 0.0025 | Obj : 0.0487 | Class : 0.0048 | Valid : 0.4208

Epoch 5/25
Train Loss : 0.0608 | Box : 0.0036 | Obj : 0.0528 | Class : 0.0045 | Valid : 0.0675

Epoch 6/25
Train Loss : 0.1508 | Box : 0.0043 | Obj : 0.1415 | Class : 0.0050 | Valid : 0.0571

Epoch 7/25
Train Loss : 0.0437 | Box : 0.0051 | Obj : 0.0346 | Class : 0.0040 | Valid : 0.0573

Epoch 8/25
Train Loss : 0.1761 | Box : 0.0066 | Obj : 0.1649 | Class : 0.0046 | Valid : 0.0489

Epoch 9/25
Train Loss : 0.0434 | Box : 0.0080 | Obj : 0.0319 | Class : 0.0035 | Valid : 0.0746

Epoch 10/25
Train Loss : 0.0416 | Box : 0.0075 | Obj : 0.0306 | Class : 0.0035 | Valid : 0.0921

Epoch 11/25
Train Loss : 0.0428 | Box 

In [ ]:
# ==========================================================
# Save Model
# ==========================================================

detector.save(
    PROJECT_ROOT / "models" / "digit_detector.keras"
)